# CIMA → LastLight

Este notebook descarga una muestra de fichas técnicas oficiales de AEMPS/CIMA, la convierte en un Knowledge Pack v1 y genera casos JSONL para `lastlight-bench`. **No es consejo médico.**

In [ ]:
from pathlib import Path
import json

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
ROOT

## 1. Revisar la selección
La muestra predeterminada contiene 10 fichas por cada uno de tres grupos ATC.

In [ ]:
config = json.loads((ROOT / 'config' / 'categories.json').read_text(encoding='utf-8'))
config

## 2. Construir el pack
La primera ejecución necesita internet. Las respuestas originales se guardan en `data/raw/cima` y se reutilizan después.

In [ ]:
import sys
sys.path.insert(0, str(ROOT / 'src'))
from cima_extractor import build_project

result = build_project(ROOT)
result

## 3. Inspeccionar el contrato
Comprobamos el manifiesto, la distribución por carpetas y los primeros casos del benchmark.

In [ ]:
pack = Path(result['pack'])
manifest = json.loads((pack / 'lastlight-pack.json').read_text(encoding='utf-8'))
markdown_files = sorted(pack.rglob('*.md'))
print('Documentos:', len(markdown_files))
print('Proveniencia:', len(manifest['provenance']))
print('Primer documento:', markdown_files[0].relative_to(pack))
print(markdown_files[0].read_text(encoding='utf-8')[:1200])

In [ ]:
eval_path = Path(result['dataset'])
cases = [json.loads(line) for line in eval_path.read_text(encoding='utf-8').splitlines()]
cases[:3]

## 4. Consulta opcional con LastLight
Instala `lastlight` y ejecuta la celda. El resultado recupera texto de fuente; no genera ni valida una recomendación médica.

In [ ]:
try:
    from lastlight import LastLight
    engine = LastLight(result['archive'], language='es', strategy='bm25')
    answer = engine.query(cases[0]['query'], top_k=3)
    print('Aceptada:', answer.accepted)
    print('Fuente:', answer.sources[0].path if answer.sources else None)
    print('Pasaje:', answer.passage)
except ImportError:
    print('Instala LastLight con: python -m pip install lastlight')